# Posterior re-derivation — schedule-agnostic GMFlow update

This notebook re-derives the GMFlow posterior step in **information form** so
the existing linear-schedule code generalises to arbitrary `(α_t, σ_t)`
schedules (VP/trig, EDM, etc.).

Companion artifacts:
- `posterior_rederivation.nb` — Mathematica derivation (canonical math source)
- `02_mathematica_crosscheck.md` — human-readable cross-check
- `_run_sympy.py`, `_run_numerical.py` — script form of this notebook
- `_test_*.py` — production-code tests in this folder

The cells here mirror the script form. Each cell prints what is verifiable so
running the notebook reproduces the evidence in `03_go_no_go.md`.


## Section 1 — Symbol setup

In [1]:
from sympy import *

alpha_t, sigma_t = symbols('alpha_t sigma_t', positive=True)
alpha_s, sigma_s = symbols('alpha_s sigma_s', positive=True)
x_t, x_s = symbols('x_t x_s', real=True)
mu_k, var_k = symbols('mu_k var_k', real=True, positive=True)

## Section 2 — Information-form posterior terms

Treating the GM component prior `N(x0; μ_k, var_k)` together with the
likelihood ratio `p(x_t|x0)/p(x_s|x0)` as a Gaussian-form correction, the
per-component posterior is again Gaussian with information-form parameters
`A = 1/(2·var_k) + ζ/2`, `B = μ_k/var_k + ν` where:


In [2]:
zeta    = (alpha_t/sigma_t)**2 - (alpha_s/sigma_s)**2
nu      = alpha_t*x_t/sigma_t**2 - alpha_s*x_s/sigma_s**2
denom_k = var_k * zeta + 1
out_mean_k        = (var_k * nu + mu_k) / denom_k
logweight_delta_k = mu_k * (nu - Rational(1, 2) * zeta * mu_k) / denom_k

print('zeta       :', simplify(zeta))
print('nu         :', simplify(nu))
print('denom_k    :', simplify(denom_k))
print('out_mean_k :', simplify(out_mean_k))
print('logw_delta :', simplify(logweight_delta_k))

zeta       : -alpha_s**2/sigma_s**2 + alpha_t**2/sigma_t**2
nu         : -alpha_s*x_s/sigma_s**2 + alpha_t*x_t/sigma_t**2
denom_k    : -alpha_s**2*var_k/sigma_s**2 + alpha_t**2*var_k/sigma_t**2 + 1
out_mean_k : (-mu_k*sigma_s**2*sigma_t**2 + var_k*(alpha_s*sigma_t**2*x_s - alpha_t*sigma_s**2*x_t))/(-sigma_s**2*sigma_t**2 + var_k*(alpha_s**2*sigma_t**2 - alpha_t**2*sigma_s**2))
logw_delta : mu_k*(2*alpha_s*sigma_t**2*x_s - 2*alpha_t*sigma_s**2*x_t - mu_k*(alpha_s**2*sigma_t**2 - alpha_t**2*sigma_s**2))/(2*(-sigma_s**2*sigma_t**2 + var_k*(alpha_s**2*sigma_t**2 - alpha_t**2*sigma_s**2)))


## Section 2.5 — Completing-the-square (load-bearing claim)

The shipped JIT uses the *truncated* `ΔLogW`, not the full log-normaliser. The
correctness argument: the dropped term is **k-independent** when `var_k` is
shared across mixture components, so it cancels under softmax.

This is the entire reason the refactor is correct. Verifying it in SymPy:


In [3]:
x0 = symbols('x0', real=True)
logQ = -(x0 - mu_k)**2 / (2 * var_k) - zeta/2 * x0**2 + nu * x0
logQ = expand(logQ)

A  = logQ.coeff(x0, 2)   # = -1/(2 var_k) - zeta/2
B  = logQ.coeff(x0, 1)   # = mu_k/var_k + nu
C0 = logQ.coeff(x0, 0)   # = -mu_k**2/(2 var_k)

logZ_full   = simplify(-B**2 / (4*A) + C0)
missingTerm = simplify(logZ_full - logweight_delta_k)
match_diff  = simplify(missingTerm - var_k * nu**2 / (2 * denom_k))

print('missingTerm                       =', missingTerm)
print('missingTerm - var_k*nu**2/(2*dk)  =', match_diff,         '   (must be 0)')
print('mu_k in missingTerm.free_symbols  =', mu_k in missingTerm.free_symbols, '(must be False)')

missingTerm                       = var_k*(-alpha_s**2*sigma_t**4*x_s**2 + 2*alpha_s*alpha_t*sigma_s**2*sigma_t**2*x_s*x_t - alpha_t**2*sigma_s**4*x_t**2)/(2*sigma_s**2*sigma_t**2*(alpha_s**2*sigma_t**2*var_k - alpha_t**2*sigma_s**2*var_k - sigma_s**2*sigma_t**2))
missingTerm - var_k*nu**2/(2*dk)  = 0    (must be 0)
mu_k in missingTerm.free_symbols  = False (must be False)


> **Historical note**: an earlier version of `posterior_rederivation.nb` used
> a bare symbol `v` instead of `ν` in `logQ`, which made `FreeQ` return
> `False` and silently broke the entire k-independence claim. Five reviewers
> all called the math correct without anyone running the notebook. The fix is
> a one-symbol substitution; both the `.nb` and this notebook now agree.


## Section 3 — Linear schedule reduction (α = 1 − σ)

Substituting the linear schedule must reproduce the existing-code expressions
literally. The operator order matters for bit-exactness — `ν` in production is
computed as `(α/σ)·x/σ` (two divides, one multiply), not `α·x/σ²`. We carry
that exact form in `nu_code` below.


In [4]:
linSubs  = {alpha_t: 1 - sigma_t, alpha_s: 1 - sigma_s}
zeta_lin = simplify(zeta.subs(linSubs))
nu_lin   = simplify(nu.subs(linSubs))

aos_t     = (1 - sigma_t) / sigma_t
aos_s     = (1 - sigma_s) / sigma_s
zeta_code = aos_t**2 - aos_s**2
nu_code   = aos_t * x_t / sigma_t - aos_s * x_s / sigma_s

print('zeta diff (must be 0):', simplify(zeta_lin - zeta_code))
print('nu   diff (must be 0):', simplify(nu_lin   - nu_code))

zeta diff (must be 0): 0


nu   diff (must be 0): 0


## Section 4 — VP / trig specialisation

For the variance-preserving cosine schedule (`α = cos(πt/2)`, `σ = sin(πt/2)`)
we check the boundary limits of the posterior mean.

**Degeneracy at `var_k = 1`**: the `t → 1⁻` limit returns
`(√2·var_k·x_s − μ_k)/(var_k − 1)`, which diverges when `var_k = 1` exactly.
This is structural and the production JIT has no guard. Worth a regression
test before any VP rollout.


In [5]:
t, s = symbols('t s', positive=True)
vpSubs = {
    alpha_t: cos(pi*t/2), sigma_t: sin(pi*t/2),
    alpha_s: cos(pi*s/2), sigma_s: sin(pi*s/2),
}
zeta_trig = trigsimp(zeta.subs(vpSubs))
vp_id     = simplify(cos(pi*t/2)**2 + sin(pi*t/2)**2 - 1)

mt    = out_mean_k.subs(vpSubs)
limT0 = simplify(limit(mt.subs(s, Rational(1, 2)), t, 0, '+'))
limT1 = simplify(limit(mt.subs(s, Rational(1, 2)), t, 1, '-'))

print('zeta_trig       :', zeta_trig)
print('cos^2+sin^2-1   :', vp_id, '   (must be 0)')
print('lim t->0 (s=1/2):', limT0, '   (expect x_t)')
print('lim t->1 (s=1/2):', limT1, '   (degenerate when var_k = 1)')

zeta_trig       : 2*cos(pi*t/2)**2/(1 - cos(pi*t)) - 2*cos(pi*s/2)**2/(1 - cos(pi*s))
cos^2+sin^2-1   : 0    (must be 0)
lim t->0 (s=1/2): x_t    (expect x_t)
lim t->1 (s=1/2): (-mu_k + sqrt(2)*var_k*x_s)/(var_k - 1)    (degenerate when var_k = 1)


## Section 5 — LaTeX export (for paper / docs)

In [6]:
print('zeta :', latex(zeta))
print('nu   :', latex(nu))
print('denom:', latex(denom_k))
print('mean :', latex(out_mean_k))
print('logw :', latex(logweight_delta_k))

zeta : - \frac{\alpha_{s}^{2}}{\sigma_{s}^{2}} + \frac{\alpha_{t}^{2}}{\sigma_{t}^{2}}
nu   : - \frac{\alpha_{s} x_{s}}{\sigma_{s}^{2}} + \frac{\alpha_{t} x_{t}}{\sigma_{t}^{2}}
denom: var_{k} \left(- \frac{\alpha_{s}^{2}}{\sigma_{s}^{2}} + \frac{\alpha_{t}^{2}}{\sigma_{t}^{2}}\right) + 1
mean : \frac{\mu_{k} + var_{k} \left(- \frac{\alpha_{s} x_{s}}{\sigma_{s}^{2}} + \frac{\alpha_{t} x_{t}}{\sigma_{t}^{2}}\right)}{var_{k} \left(- \frac{\alpha_{s}^{2}}{\sigma_{s}^{2}} + \frac{\alpha_{t}^{2}}{\sigma_{t}^{2}}\right) + 1}
logw : \frac{\mu_{k} \left(- \frac{\alpha_{s} x_{s}}{\sigma_{s}^{2}} + \frac{\alpha_{t} x_{t}}{\sigma_{t}^{2}} - \mu_{k} \left(- \frac{\alpha_{s}^{2}}{2 \sigma_{s}^{2}} + \frac{\alpha_{t}^{2}}{2 \sigma_{t}^{2}}\right)\right)}{var_{k} \left(- \frac{\alpha_{s}^{2}}{\sigma_{s}^{2}} + \frac{\alpha_{t}^{2}}{\sigma_{t}^{2}}\right) + 1}


## Section 6 — Numerical verification

Three numerical tests, each independent of the symbolic derivation:

1. **Linear-schedule equivalence**: the general formula at `α = 1 − σ`
   must agree with the existing JIT to floating-point precision.
2. **Trig-schedule 1D exactness**: at `C = 1, K = 4`, the analytic posterior
   mean must match a 1D brute-force grid integration.
3. **Float32 stability**: at very small `t`, the VP `ζ` grows quickly; we
   verify the analytic `zeta_max` clamp prevents float32 overflow.


In [7]:
import torch
import math

torch.manual_seed(42)
B, K, C = 2, 4, 8


def posterior_general(alpha_t, sigma_t, alpha_s, sigma_s,
                      x_t, x_s, gm_means, gm_vars, gm_logweights,
                      eps=1e-6, ZETA_MAX=None):
    sigma_t = sigma_t.clamp(min=eps)
    sigma_s = sigma_s.clamp(min=eps)
    zeta = (alpha_t / sigma_t).square() - (alpha_s / sigma_s).square()
    nu   = alpha_t * x_t / sigma_t**2 - alpha_s * x_s / sigma_s**2
    if ZETA_MAX is not None:
        zeta = zeta.clamp(max=ZETA_MAX)
    nu, zeta   = nu.unsqueeze(1), zeta.unsqueeze(1)
    denom      = (gm_vars * zeta + 1).clamp(min=eps)
    out_means  = (gm_vars * nu + gm_means) / denom
    logw_delta = (gm_means * (nu - 0.5 * zeta * gm_means)).sum(
        dim=-1, keepdim=True) / denom
    weights    = (gm_logweights + logw_delta).softmax(dim=1)
    return (out_means * weights).sum(dim=1)


def ref_linear(sigma_s, sigma_t, x_s, x_t, gm_means, gm_vars, gm_logweights, eps=1e-6):
    sigma_s, sigma_t = sigma_s.clamp(min=eps), sigma_t.clamp(min=eps)
    aos_s, aos_t = (1 - sigma_s) / sigma_s, (1 - sigma_t) / sigma_t
    zeta = aos_t.square() - aos_s.square()
    nu   = aos_t * x_t / sigma_t - aos_s * x_s / sigma_s
    nu, zeta   = nu.unsqueeze(1), zeta.unsqueeze(1)
    denom      = (gm_vars * zeta + 1).clamp(min=eps)
    out_means  = (gm_vars * nu + gm_means) / denom
    logw_delta = (gm_means * (nu - 0.5 * zeta * gm_means)).sum(
        dim=-1, keepdim=True) / denom
    weights    = (gm_logweights + logw_delta).softmax(dim=1)
    return (out_means * weights).sum(dim=1)


gm_means      = torch.randn(B, K, C)
gm_logweights = torch.randn(B, K, 1)
gm_vars       = torch.rand(B, 1, 1).abs() + 0.1
x_s, x_t      = torch.randn(B, C), torch.randn(B, C)

sigma_s_val = (torch.rand(B, 1) * 0.4 + 0.4)
sigma_t_val = (sigma_s_val * torch.rand(B, 1) * 0.6).clamp(min=1e-3)
alpha_s_lin = 1 - sigma_s_val
alpha_t_lin = 1 - sigma_t_val

out_a = posterior_general(alpha_t_lin, sigma_t_val, alpha_s_lin, sigma_s_val,
                          x_t, x_s, gm_means, gm_vars, gm_logweights)
out_b = ref_linear(sigma_s_val, sigma_t_val, x_s, x_t,
                   gm_means, gm_vars, gm_logweights)

diff_ab = (out_a - out_b).abs().max().item()
print(f'[Test 1] linear schedule max |diff|: {diff_ab:.2e}  -- expect ~1e-7')
assert diff_ab < 1e-5, f'FAIL: {diff_ab}'
print('PASS: general formula at alpha = 1 - sigma matches existing-code reference')

[Test 1] linear schedule max |diff|: 2.38e-07  -- expect ~1e-7
PASS: general formula at alpha = 1 - sigma matches existing-code reference


### Test 2 — Trig (VP) 1D analytic vs brute-force

At `C = 1`, `K = 4`, the analytic posterior mean must match a 1D grid
integration of the importance-reweighted GM. This is the *only* end-to-end
numerical test of the new VP code path. The `_run_numerical.py` script
historically reports `|diff| ≈ 1.77e-07` here.


In [8]:
t_val = torch.rand(B, 1) * 0.4 + 0.3
s_val = (t_val + torch.rand(B, 1) * 0.2 + 0.1).clamp(max=0.95)

alpha_t_trig = torch.cos(math.pi * t_val / 2)
sigma_t_trig = torch.sin(math.pi * t_val / 2)
alpha_s_trig = torch.cos(math.pi * s_val / 2)
sigma_s_trig = torch.sin(math.pi * s_val / 2)

x0_true = torch.randn(B, C)
x_t_vp  = alpha_t_trig * x0_true + sigma_t_trig * torch.randn(B, C)
x_s_vp  = alpha_s_trig * x0_true + sigma_s_trig * torch.randn(B, C)


def brute_force_mean_1d(at, st, as_, ss, mu_k_vals, var_k_val, logw_k_vals,
                         xt, xs, n_grid=20_000, std_range=12):
    K = len(mu_k_vals)
    at, st, as_, ss = float(at), float(st), float(as_), float(ss)
    xt, xs, vk      = float(xt), float(xs), float(var_k_val)

    nu_sc   = at*xt/st**2 - as_*xs/ss**2
    zeta_sc = at**2/st**2 - as_**2/ss**2
    center  = (float(mu_k_vals[0]) + vk*nu_sc) / (vk*zeta_sc + 1)
    half    = std_range * (vk**0.5 + st / max(at, 1e-8))
    grid    = torch.linspace(center - half, center + half, n_grid)

    log_gm = torch.stack([
        float(logw_k_vals[k]) - 0.5*(grid - float(mu_k_vals[k]))**2 / vk
        for k in range(K)
    ]).logsumexp(dim=0)
    log_ratio = (-0.5*(xt - at*grid)**2/st**2) - (-0.5*(xs - as_*grid)**2/ss**2)
    log_post  = log_gm + log_ratio
    log_post  = log_post - log_post.logsumexp(dim=0)
    return float((log_post.exp() * grid).sum())


b, c = 0, 0
mu_k_1d  = gm_means[b, :, c]
logw_1d  = gm_logweights[b, :, 0]
var_k_1d = gm_vars[b, 0, 0]
at_b, st_b = float(alpha_t_trig[b, 0]), float(sigma_t_trig[b, 0])
as_b, ss_b = float(alpha_s_trig[b, 0]), float(sigma_s_trig[b, 0])

nu_sc   = at_b * float(x_t_vp[b, c]) / st_b**2 - as_b * float(x_s_vp[b, c]) / ss_b**2
zeta_sc = at_b**2 / st_b**2 - as_b**2 / ss_b**2
denom_1d = float(var_k_1d) * zeta_sc + 1
means_post = [(float(var_k_1d) * nu_sc + float(mu_k_1d[k])) / denom_1d for k in range(K)]
logw_delta = [float(mu_k_1d[k]) * (nu_sc - 0.5 * zeta_sc * float(mu_k_1d[k])) / denom_1d
              for k in range(K)]
logw_post = torch.tensor([float(logw_1d[k]) + logw_delta[k] for k in range(K)])
w_post    = torch.softmax(logw_post, dim=0)
mean_analytic = float(sum(float(w_post[k]) * means_post[k] for k in range(K)))

mean_brute = brute_force_mean_1d(at_b, st_b, as_b, ss_b,
                                  mu_k_1d, var_k_1d, logw_1d,
                                  x_t_vp[b, c], x_s_vp[b, c])
diff_1d = abs(mean_analytic - mean_brute)
print(f'[Test 2] analytic={mean_analytic:.6f}  brute={mean_brute:.6f}  |diff|={diff_1d:.2e}')
assert diff_1d < 0.01, f'FAIL: {diff_1d}'
print('PASS: trig posterior formula matches 1D brute-force (C=1, K=4)')

[Test 2] analytic=1.913505  brute=1.913505  |diff|=1.77e-07
PASS: trig posterior formula matches 1D brute-force (C=1, K=4)


### Test 3 — Float32 stability + analytic `zeta_max` bound

For VP near `t = 0`, `ζ ≈ (2/(πt))²` grows quadratically. Without a clamp,
`var_k · ζ` overflows float32 around `t ≈ 1e-19` for `var_k = 10`. The clamp
`zeta_max = float32_max / max(var_k)` keeps `denom = var_k·ζ + 1` finite.

> **Caveat (general's empirical finding)**: the shipped
> `gmflow_posterior_mean_jit_general` lacks this clamp. In practice, when
> `var_k · ζ` overflows symmetrically in numerator and denominator, the ratio
> still resolves finitely. So today's code is empirically safe, but for
> *fragile* reasons (relying on undocumented IEEE-754 cancellation). The
> clamp should be ported into production as engineering hygiene before VP
> ships. See `_test_vp_stability.py`.


In [9]:
float32_max   = torch.finfo(torch.float32).max
max_var_k     = 10.0
ZETA_MAX_safe = float32_max / max_var_k
t_min_safe    = 2.0 / (math.pi * math.sqrt(ZETA_MAX_safe))
print(f'float32 max:          {float32_max:.3e}')
print(f'ZETA_MAX_safe:        {ZETA_MAX_safe:.3e}  (= float32_max / max_var_k={max_var_k})')
print(f'min safe t (no clamp):{t_min_safe:.3e}')

t_tiny  = torch.tensor([[1e-5]])
s_mid   = torch.tensor([[0.5]])
alpha_t_, sigma_t_ = torch.cos(math.pi * t_tiny / 2), torch.sin(math.pi * t_tiny / 2)
alpha_s_, sigma_s_ = torch.cos(math.pi * s_mid  / 2), torch.sin(math.pi * s_mid  / 2)

zeta_raw = (alpha_t_ / sigma_t_.clamp(min=1e-8)).square() - (alpha_s_ / sigma_s_).square()
zeta_clamped = zeta_raw.clamp(max=ZETA_MAX_safe)
denom_test = torch.tensor([[[max_var_k]]]) * zeta_clamped + 1

print(f'\nzeta at t=1e-5 raw:   {zeta_raw.item():.3e}  isinf={torch.isinf(zeta_raw).item()}')
print(f'zeta after clamp:    {zeta_clamped.item():.3e}')
print(f'denom (var*zeta+1):  {denom_test.item():.3e}  isinf={torch.isinf(denom_test).item()}')
print('PASS: clamp keeps denom finite' if not torch.isinf(denom_test) else 'FAIL')

print('\nSNR(t) for VP vs VE schedules:')
print(f'{"t":>8}  {"VP SNR":>14}  {"VE SNR":>14}')
for tv in [1e-3, 0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    snr_vp = (math.cos(math.pi*tv/2) / math.sin(math.pi*tv/2))**2
    snr_ve = ((1 - tv) / tv)**2
    print(f'{tv:>8.3f}  {snr_vp:>14.2f}  {snr_ve:>14.2f}')

float32 max:          3.403e+38
ZETA_MAX_safe:        3.403e+37  (= float32_max / max_var_k=10.0)
min safe t (no clamp):1.091e-19

zeta at t=1e-5 raw:   4.053e+09  isinf=False
zeta after clamp:    4.053e+09
denom (var*zeta+1):  4.053e+10  isinf=False
PASS: clamp keeps denom finite

SNR(t) for VP vs VE schedules:
       t          VP SNR          VE SNR
   0.001       405284.07       998001.00
   0.010         4052.18         9801.00
   0.100           39.86           81.00
   0.300            3.85            5.44
   0.500            1.00            1.00
   0.700            0.26            0.18
   0.900            0.03            0.01
   0.990            0.00            0.00


## Summary

| Layer | Tool | Result |
|---|---|---|
| Symbolic identity (`vk·ν²/(2·dk)` k-independent) | SymPy + Mathematica | `0` / `False` for `μ_k ∈ free_symbols` |
| Linear schedule reduction | SymPy | `ζ_lin == ζ_code`, `ν_lin == ν_code` |
| VP boundary at `t → 0⁺` | SymPy | `x_t` (correct) |
| VP boundary at `t → 1⁻` | SymPy | diverges at `var_k = 1` (regression-test gap) |
| Linear formula vs production | torch float32 | `|diff| ≤ 1e-6` |
| Trig 1D analytic vs brute | torch grid | `|diff| ≈ 1.77e-07` |
| Float32 VP stability with clamp | torch | `denom` finite at `t = 1e-5` |

For the production code-path tests, see `_test_differential_equivalence.py`,
`_test_vp_stability.py`, and `_test_wrapper_dispatch.py` in this folder.
